# Build DPO Pairs: Conditional Neutral-as-Rejected (80/20 Regularization)

Pipeline:
1. Load positive samples from ./data/llm_train.json.
2. For each sample, generate 1 candidate with the V4.3 style model (no lexical signature hints).
3. Check rewrite distance: `SequenceMatcher(model_output_no_lex, neutral_sentence).ratio()`.
   - If ratio > 0.75 (rewrite insufficient):
     - 80% → use `neutral_sentence` as rejected (barely rewritten).
     - 20% → keep model output as rejected (regularization against over-correction).
   - If ratio ≤ 0.75 (rewrite sufficient): keep model output as rejected.
4. Fallback: if generated candidate is invalid (empty output / CoT too short), use `neutral_sentence`.
5. Export DPO pairs to JSON and JSONL.

DPO signal: chosen = fully stylized output, rejected = barely rewritten (when rewrite distance is insufficient).

In [1]:
# Run this on Remote Jupyter Book.

import os

os.getcwd()
os.chdir("/root/OtakuLab")

In [3]:
import json
import random
import re
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Optional, TypedDict, cast

import numpy as np
import torch
import torch.nn as nn
from peft import PeftModel
from tqdm import tqdm
from transformers import AutoTokenizer, Qwen3ForCausalLM

POSITIVE_DATA_PATH = Path("./data/llm_train.json")
DPO_OUTPUT_JSONL_PATH = Path("./data/dpo_pairs_wo_cot_shared.jsonl")
LOST_SAMPLE_OUTPUT_JSON_PATH = Path("./data/dpo_pairs_wo_cot_shared_lost_samples.json")

BASE_MODEL_PATH = Path("/root/autodl-tmp/Qwen3-1.7B/")
V43_CHECKPOINT_DIR = Path("/root/OtakuLab/outputs/styled-qwen-v4.3-4e-05")

NUM_CANDIDATES = 1
NUM_ROUNDS = 1
TEMPERATURE = 0.3
TOP_P = 0.9
MAX_NEW_TOKENS = 768
REPETITION_PENALTY = 1.05
ENABLE_THINKING = True
FORCE_INJECT_CHINESE_ANCHOR = True
MIN_COT_CHARS = 8
GEN_BATCH_SIZE = 80

NEUTRAL_REJECT_RATIO = 0.8
REWRITE_DISTANCE_THRESHOLD = 0.85
USE_RAW_MODEL_REJECT = True
"""设置 True 以将 SFT 模型的输出作为 Reject"""

MAX_SAMPLES: Optional[int] = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Base model:", BASE_MODEL_PATH)
print("V4.3 checkpoint:", V43_CHECKPOINT_DIR)
print("Generation batch size:", GEN_BATCH_SIZE)
print("Neutral reject ratio:", NEUTRAL_REJECT_RATIO)
print("Rewrite distance threshold:", REWRITE_DISTANCE_THRESHOLD)
print("Use raw model reject:", USE_RAW_MODEL_REJECT)

Device: cuda
Base model: /root/autodl-tmp/Qwen3-1.7B
V4.3 checkpoint: /root/OtakuLab/outputs/styled-qwen-v4.3-4e-05
Generation batch size: 80
Neutral reject ratio: 0.8
Rewrite distance threshold: 0.85
Use raw model reject: True


## Load Positive Samples and Prompt Builders

In [4]:
SYNTACTIC_DIMS = [
    "mdd",
    "verb_noun_ratio",
    "question_ratio",
    "imperative_ratio",
    "fragment_ratio",
    "marker_density",
]

EN_NAME_TO_ZH = {
    "Muice": "沐雪",
    "Ayaka": "神里绫华",
    "Zhongli": "钟离",
    "Hutao": "胡桃",
    "Haruhi": "凉宫春日",
    "Liyunlong": "李云龙",
    "Sheldon": "谢尔顿",
    "Wukong": "孙悟空",
}
ZH_NAME_TO_EN = {v: k for k, v in EN_NAME_TO_ZH.items()}

class InstructionComponents(TypedDict):
    semantic_tags: list[str]
    structural_metrics: dict[str, float]
    signature_rules: list[str]

class PositiveItem(TypedDict):
    character: str
    neutral_sentence: str
    instruction_components: InstructionComponents
    thinking_process: str
    output: str

def build_system_prompt() -> str:
    return (
        "You are an expert in stylistic rewriting. "
        "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
    )

def build_user_prompt(item: PositiveItem) -> str:
    instr = item["instruction_components"]
    semantic_tags = instr.get("semantic_tags", [])
    semantic_text = ", ".join(semantic_tags) if semantic_tags else "None"
    signature_text = "None"  # 剥夺词汇提示以降低模型的风格能力
    return (
        f"Target Character: {item['character']}\n"
        f"Semantic Traits: {semantic_text}\n"
        f"Formatting Signature: {signature_text}\n"
        f"Neutral Content: {item['neutral_sentence']}\n"
    )

def compose_assistant_text(cot_text: str, output_text: str) -> str:
    cot = (cot_text or "").strip().replace("<think>", "").replace("</think>", "")
    out = (output_text or "").strip()
    if cot:
        return f"<think>{cot}</think>\n\n{out}".strip()
    return out

def load_positive_items(path: Path) -> list[PositiveItem]:
    if not path.exists():
        raise FileNotFoundError(f"Missing positive data file: {path}")

    raw_data = json.loads(path.read_text(encoding="utf-8"))
    items: list[PositiveItem] = []

    for row in raw_data:
        character = str(row.get("character", "")).strip()
        neutral_sentence = str(row.get("neutral_sentence", "")).strip()
        output = str(row.get("output", "")).strip()
        thinking_process = str(row.get("thinking_process", "")).strip()
        instr_raw = row.get("instruction_components", {})

        if not character or not neutral_sentence or not output or not isinstance(instr_raw, dict):
            continue

        structural_metrics_raw = instr_raw.get("structural_metrics", {})
        if not isinstance(structural_metrics_raw, dict):
            continue

        structural_metrics = {
            dim: float(structural_metrics_raw.get(dim, 0.0))
            for dim in SYNTACTIC_DIMS
        }

        semantic_tags = instr_raw.get("semantic_tags", [])
        signature_rules = instr_raw.get("signature_rules", [])

        if not isinstance(semantic_tags, list):
            semantic_tags = []
        if not isinstance(signature_rules, list):
            signature_rules = []

        item: PositiveItem = {
            "character": character,
            "neutral_sentence": neutral_sentence,
            "instruction_components": {
                "semantic_tags": [str(x) for x in semantic_tags],
                "structural_metrics": structural_metrics,
                "signature_rules": [str(x) for x in signature_rules],
            },
            "thinking_process": thinking_process,
            "output": output,
        }
        items.append(item)

    return items

positive_items = load_positive_items(POSITIVE_DATA_PATH)
if MAX_SAMPLES is not None:
    positive_items = positive_items[:MAX_SAMPLES]

print(f"Loaded positive samples: {len(positive_items)}")

Loaded positive samples: 8272


## Load V4.3 Style Model (for generated rejects)

In [5]:
class StyleEncoder(nn.Module):
    def __init__(self, input_dim: int, out_dim: int):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, out_dim // 2),
            nn.ReLU(),
            nn.Linear(out_dim // 2, out_dim),
            nn.Tanh(),
        )

    def forward(self, syntactic_vec: torch.Tensor) -> torch.Tensor:
        return self.proj(syntactic_vec)

def ensure_generator_assets(base_model_path: Path, checkpoint_dir: Path) -> None:
    missing = []
    if not base_model_path.exists():
        missing.append(str(base_model_path))

    required_paths = [
        checkpoint_dir / "lora",
        checkpoint_dir / "style_encoder.pt",
        checkpoint_dir / "tokenizer",
    ]
    for p in required_paths:
        if not p.exists():
            missing.append(str(p))

    if missing:
        raise FileNotFoundError(
            "Missing generator assets. Please verify V4.3 paths:\n" + "\n".join(missing)
        )

ensure_generator_assets(BASE_MODEL_PATH, V43_CHECKPOINT_DIR)

gen_tokenizer = AutoTokenizer.from_pretrained(V43_CHECKPOINT_DIR / "tokenizer")
if gen_tokenizer.pad_token is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token
gen_tokenizer.padding_side = "right"

base_model = Qwen3ForCausalLM.from_pretrained(BASE_MODEL_PATH, dtype=torch.bfloat16)
gen_model = PeftModel.from_pretrained(base_model, V43_CHECKPOINT_DIR / "lora")

hidden_size_raw = base_model.config.hidden_size
if hidden_size_raw is None:
    raise ValueError("base model hidden_size is None")
hidden_size = int(hidden_size_raw)

style_encoder = StyleEncoder(len(SYNTACTIC_DIMS), hidden_size)
style_encoder.load_state_dict(torch.load(V43_CHECKPOINT_DIR / "style_encoder.pt", map_location=DEVICE))

style_encoder.to(DEVICE)
gen_model.to(DEVICE)
style_encoder.eval()
gen_model.eval()

GEN_DTYPE = next(gen_model.parameters()).dtype
print("Generator loaded. dtype =", GEN_DTYPE)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Generator loaded. dtype = torch.bfloat16


## Candidate Generation and Conditional Pair Assembly

In [6]:
# pyright: reportCallIssue=false

def clean_candidate_text(text: str) -> tuple[str, int]:
    raw = (text or "").strip()
    if not raw:
        return "", 0

    kept: list[str] = []
    for line in raw.splitlines():
        s = line.strip()
        if not s:
            continue

        low = s.lower()
        if low.startswith("target character:"):
            continue
        if low.startswith("semantic traits:"):
            continue
        if low.startswith("formatting signature:"):
            continue
        if low.startswith("neutral content:"):
            continue

        kept.append(s)

    cleaned = "\n".join(kept).strip()
    cleaned = re.sub(
        r"^(assistant|response|answer|回答)\s*[:：]\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    ).strip()

    removed_chars = max(0, len(raw) - len(cleaned))
    return cleaned, removed_chars


def strip_prompt_prefix(decoded_text: str, prompt_text: str) -> str:
    text = decoded_text.strip()
    prompt = prompt_text.strip()
    if text.startswith(prompt):
        return text[len(prompt):].strip()

    for window in (256, 128, 64, 32):
        if len(prompt) >= window:
            tail = prompt[-window:]
            if text.startswith(tail):
                return text[window:].strip()
    return text


def extract_cot_and_output(generated_text: str) -> tuple[str, str, dict[str, Any]]:
    text = generated_text.strip()

    think_match = re.search(r"<think>(.*?)</think>", text, flags=re.IGNORECASE | re.DOTALL)
    has_think_tag = think_match is not None

    if think_match:
        raw_cot = think_match.group(1).strip()
        raw_output = text[think_match.end():].strip()
    else:
        raw_cot = ""
        raw_output = text

    cot_text, cot_removed_chars = clean_candidate_text(raw_cot)
    output_text, output_removed_chars = clean_candidate_text(raw_output)

    # 保守策略：若 CoT 被清洗为空，回退到原始 CoT，避免过度清洗丢失推理文本。
    if raw_cot and not cot_text:
        cot_text = raw_cot

    diagnostics = {
        "has_think_tag": has_think_tag,
        "raw_cot": raw_cot,
        "raw_output": raw_output,
        "cot_removed_chars": cot_removed_chars,
        "output_removed_chars": output_removed_chars,
        "cleaning_removed_chars": cot_removed_chars + output_removed_chars,
    }
    return cot_text, output_text, diagnostics


@torch.inference_mode()
def generate_candidates_batch(items: list[PositiveItem], temperature: float) -> list[dict[str, Any]]:
    if not items:
        return []

    system_prompt = build_system_prompt()
    prompt_texts: list[str] = []
    structural_rows: list[list[float]] = []

    for item in items:
        user_prompt = build_user_prompt(item)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        prompt_text = gen_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )

        if ENABLE_THINKING and FORCE_INJECT_CHINESE_ANCHOR:
            prompt_text = prompt_text.rstrip() + " " + "<think>思考: "

        prompt_texts.append(prompt_text)

        structural_metrics = item["instruction_components"].get("structural_metrics", {})
        structural_rows.append([float(structural_metrics.get(dim, 0.0)) for dim in SYNTACTIC_DIMS])

    # 批量生成时使用左侧填充，保证因果注意力对齐。
    original_padding_side = gen_tokenizer.padding_side
    gen_tokenizer.padding_side = "left"
    try:
        tokenized = gen_tokenizer(
            prompt_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=768,
        ).to(DEVICE)
    finally:
        gen_tokenizer.padding_side = original_padding_side

    input_ids = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]

    syntactic_tensor = torch.tensor(
        structural_rows,
        dtype=torch.float32,
        device=DEVICE,
    )

    style_dtype = next(style_encoder.parameters()).dtype
    style_emb = style_encoder(syntactic_tensor.to(style_dtype)).to(GEN_DTYPE)
    style_prefix = style_emb.unsqueeze(1)

    embedding_layer = cast(nn.Module, gen_model.get_input_embeddings())
    token_embeds = embedding_layer(input_ids)
    inputs_embeds = torch.cat([style_prefix, token_embeds], dim=1)

    prefix_mask = torch.ones((attention_mask.shape[0], 1), dtype=torch.long, device=DEVICE)
    new_attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

    generated_ids = gen_model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=new_attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=temperature,
        top_p=TOP_P,
        do_sample=True,
        repetition_penalty=REPETITION_PENALTY,
        pad_token_id=gen_tokenizer.pad_token_id,
    )

    decoded_batch = gen_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    generated_rows: list[dict[str, Any]] = []
    for decoded, prompt_text in zip(decoded_batch, prompt_texts):
        decoded = "<think>" + decoded if ENABLE_THINKING and FORCE_INJECT_CHINESE_ANCHOR else decoded
        generated_text = strip_prompt_prefix(decoded, prompt_text)
        cot_text, output_text, parse_diag = extract_cot_and_output(generated_text)

        generated_rows.append(
            {
                "prompt": prompt_text,
                "decoded": decoded,
                "generated_text": generated_text,
                "cot": cot_text,
                "output": output_text,
                **parse_diag,
            }
        )

    return generated_rows


def normalize_for_similarity(text: str) -> str:
    return "".join(ch.lower() for ch in text if ch.isalnum())


def compute_structural_similarity(text_a: str, text_b: str) -> float:
    a_norm = normalize_for_similarity(text_a)
    b_norm = normalize_for_similarity(text_b)
    if not a_norm or not b_norm:
        return 0.0
    return float(SequenceMatcher(None, a_norm, b_norm).ratio())


def is_extremely_structurally_similar(neutral_sentence: str, output: str) -> bool:
    neutral_norm = normalize_for_similarity(neutral_sentence)
    output_norm = normalize_for_similarity(output)

    if not neutral_norm or not output_norm:
        return False
    if neutral_norm == output_norm:
        return True

    matcher = SequenceMatcher(None, neutral_norm, output_norm)
    ratio = matcher.ratio()
    if ratio >= 0.93:
        return True

    shorter = min(len(neutral_norm), len(output_norm))
    longer = max(len(neutral_norm), len(output_norm))
    length_ratio = shorter / longer
    if length_ratio >= 0.86 and (neutral_norm in output_norm or output_norm in neutral_norm):
        return True

    longest_match = matcher.find_longest_match(0, len(neutral_norm), 0, len(output_norm)).size
    return ratio >= 0.88 and (longest_match / shorter) >= 0.93


def build_dpo_pairs(
    items: list[PositiveItem],
    generation_batch_size: int = GEN_BATCH_SIZE,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[str, int]]:
    generation_batch_size = max(1, int(generation_batch_size))

    stats: dict[str, int] = {
        "total": len(items),
        "paired": 0,
        "lost_total": 0,
        "strategy_neutral": 0,
        "strategy_generated": 0,
        "strategy_generated_fallback_to_neutral": 0,
        "rewrite_distance_exceeded": 0,
        "no_eligible_candidate": 0,
    }

    pairs: list[dict[str, Any]] = []
    lost_samples: list[dict[str, Any]] = []

    # --- Phase 1: Generate 1 candidate per sample ---
    candidate_map: dict[int, dict[str, Any]] = {}

    jobs: list[dict[str, Any]] = []
    for sample_idx in range(len(items)):
        jobs.append({"sample_idx": sample_idx, "round": 1, "candidate_idx": 1})

    for start in tqdm(
        range(0, len(jobs), generation_batch_size),
        desc=f"Generating candidates (bs={generation_batch_size})",
    ):
        batch_jobs = jobs[start : start + generation_batch_size]
        batch_items = [items[job["sample_idx"]] for job in batch_jobs]

        generated_rows = generate_candidates_batch(batch_items, temperature=TEMPERATURE)

        for pos, job in enumerate(batch_jobs):
            if pos >= len(generated_rows) or generated_rows[pos] is None:
                continue

            generated = cast(dict[str, Any], generated_rows[pos])
            output_text = generated.get("output", "").strip()
            cot_text = generated.get("cot", "").strip()
            neutral_text = items[job["sample_idx"]]["neutral_sentence"]

            valid_output = bool(output_text)
            valid_cot = bool(len(cot_text) >= MIN_COT_CHARS)
            eligible = valid_output and valid_cot
            structural_similarity = (
                compute_structural_similarity(neutral_text, output_text)
                if valid_output else 0.0
            )

            candidate_map[job["sample_idx"]] = {
                "round": job["round"],
                "candidate_idx": job["candidate_idx"],
                "cot": cot_text,
                "output": output_text,
                "valid_output": valid_output,
                "valid_cot": valid_cot,
                "eligible": eligible,
                "structural_similarity": structural_similarity,
                "source": "generated_candidate",
                "has_think_tag": bool(generated.get("has_think_tag", False)),
                "cleaning_removed_chars": int(generated.get("cleaning_removed_chars", 0)),
                "cleaned_to_empty": bool((not valid_output) and generated.get("raw_output", "").strip()),
            }

    # --- Phase 2: Conditional pair assembly ---
    for sample_idx, item in enumerate(tqdm(items, desc="Assembling DPO pairs")):
        chosen_cot = item.get("thinking_process", "")
        chosen_output = item.get("output", "")
        neutral_sentence = item.get("neutral_sentence", "")

        chosen_neutral_structural_similarity = compute_structural_similarity(
            neutral_sentence,
            chosen_output,
        )

        candidate = candidate_map.get(sample_idx)
        candidate_valid = candidate is not None and candidate.get("eligible", False)

        if USE_RAW_MODEL_REJECT:
            # 直出模式：始终使用模型原始输出作为 rejected（无效时 fallback）
            if candidate_valid:
                assert candidate is not None
                rejected_cot = candidate.get("cot", chosen_cot)
                rejected_output = candidate.get("output", "")
                reject_source = "generated_candidate"
                stats["strategy_generated"] += 1
            else:
                rejected_cot = chosen_cot
                rejected_output = neutral_sentence
                reject_source = "neutral_sentence_fallback"
                stats["strategy_generated_fallback_to_neutral"] += 1
                stats["no_eligible_candidate"] += 1
        elif candidate_valid:
            assert candidate is not None
            rewrite_ratio = float(candidate.get("structural_similarity", 0.0))

            if rewrite_ratio > REWRITE_DISTANCE_THRESHOLD:
                # 改写幅度不足
                stats["rewrite_distance_exceeded"] += 1

                if random.random() < NEUTRAL_REJECT_RATIO:
                    # 80%: 用 neutral 作为 rejected
                    rejected_cot = chosen_cot
                    rejected_output = neutral_sentence
                    reject_source = "neutral_sentence"
                    stats["strategy_neutral"] += 1
                else:
                    # 20%: 保留原模型输出作为正则化
                    rejected_cot = candidate.get("cot",chosen_cot)
                    rejected_output = candidate.get("output", "")
                    reject_source = "generated_candidate"
                    stats["strategy_generated"] += 1
            else:
                # 改写幅度足够，保持原逻辑
                rejected_cot = candidate.get("cot", chosen_cot)
                rejected_output = candidate.get("output", "")
                reject_source = "generated_candidate"
                stats["strategy_generated"] += 1
        else:
            # 候选无效，fallback 到 neutral
            rejected_cot = chosen_cot
            rejected_output = neutral_sentence
            reject_source = "neutral_sentence_fallback"
            stats["strategy_generated_fallback_to_neutral"] += 1

            if not candidate_valid:
                stats["no_eligible_candidate"] += 1

        pair = {
            "prompt": [
                {"role": "system", "content": build_system_prompt()},
                {"role": "user", "content": build_user_prompt(item)},
            ],
            "chosen": compose_assistant_text(chosen_cot, chosen_output),
            "rejected": compose_assistant_text(rejected_cot, rejected_output),
            "meta": {
                "character": item["character"],
                "reject_source": reject_source,
                "reject_strategy": "neutral" if "neutral" in reject_source else "generated",
                "reject_rewrite_ratio": float(candidate.get("structural_similarity", 0.0)) if candidate is not None else 0.0,
                "chosen_neutral_structural_similarity": float(chosen_neutral_structural_similarity),
            },
        }

        pairs.append(pair)
        stats["paired"] += 1

    return pairs, lost_samples, stats

## Build and Export DPO Dataset

> Pair construction strategy (v2: conditional neutral-as-rejected)
1. Generate 1 candidate per sample via V4.3 model (no lexical signature hints).
2. For each sample, compute `rewrite_ratio = SequenceMatcher(model_output, neutral_sentence).ratio()`.
3. If `rewrite_ratio > 0.75` (rewrite insufficient):
   - 80% → use `neutral_sentence` as rejected (barely rewritten).
   - 20% → keep model output as rejected (regularization).
4. If `rewrite_ratio ≤ 0.75` (rewrite sufficient): keep model output as rejected.
5. Fallback: if candidate is invalid (empty output / CoT too short), use `neutral_sentence`.
6. Expected: near-zero loss, ~100% pair yield.

In [7]:
pairs, lost_samples, stats = build_dpo_pairs(positive_items, generation_batch_size=GEN_BATCH_SIZE)

with DPO_OUTPUT_JSONL_PATH.open("w", encoding="utf-8") as f:
    for row in pairs:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

LOST_SAMPLE_OUTPUT_JSON_PATH.write_text(
    json.dumps(lost_samples, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("DPO build done.")
print("Stats:", stats)
if stats.get("total", 0) > 0:
    pair_ratio = stats.get("paired", 0) / stats["total"]
    lost_ratio = stats.get("lost_total", 0) / stats["total"]
    print(f"Pair ratio: {pair_ratio:.2%}")
    print(f"Lost ratio: {lost_ratio:.2%}")
print()
print("--- Strategy breakdown ---")
print(f"Rewrite distance exceeded (> {REWRITE_DISTANCE_THRESHOLD}): {stats.get('rewrite_distance_exceeded', 0)}")
print(f"Strategy neutral (80% path): {stats.get('strategy_neutral', 0)}")
print(f"Strategy generated (20% path + sufficient rewrite): {stats.get('strategy_generated', 0)}")
print(f"Fallback to neutral (invalid candidate): {stats.get('strategy_generated_fallback_to_neutral', 0)}")
print(f"No eligible candidate: {stats.get('no_eligible_candidate', 0)}")
print()
print("Saved JSONL:", DPO_OUTPUT_JSONL_PATH)
print("Saved lost JSON :", LOST_SAMPLE_OUTPUT_JSON_PATH)

Assembling DPO pairs: 100%|██████████| 8272/8272 [00:01<00:00, 7252.77it/s] 


DPO build done.
Stats: {'total': 8272, 'paired': 8272, 'lost_total': 0, 'strategy_neutral': 0, 'strategy_generated': 8247, 'strategy_generated_fallback_to_neutral': 25, 'rewrite_distance_exceeded': 0, 'no_eligible_candidate': 25}
Pair ratio: 100.00%
Lost ratio: 0.00%

--- Strategy breakdown ---
Rewrite distance exceeded (> 0.85): 0
Strategy neutral (80% path): 0
Strategy generated (20% path + sufficient rewrite): 8247
Fallback to neutral (invalid candidate): 25
No eligible candidate: 25

Saved JSONL: data/dpo_pairs_wo_cot_shared.jsonl
Saved lost JSON : data/dpo_pairs_wo_cot_shared_lost_samples.json


In [8]:
print("=== Pair Preview ===")
preview_n = min(3, len(pairs))
for i in range(preview_n):
    print(f"--- Pair {i + 1} ---")
    print("Character:", pairs[i]["meta"]["character"])
    print("Reject source:", pairs[i]["meta"]["reject_source"])
    print("Reject strategy:", pairs[i]["meta"]["reject_strategy"])
    print("Reject rewrite ratio:", pairs[i]["meta"]["reject_rewrite_ratio"])
    print("Chosen-neutral structural similarity:", pairs[i]["meta"]["chosen_neutral_structural_similarity"])
    print("Chosen output:", pairs[i]["chosen"])
    print("Rejected output:", pairs[i]["rejected"])
    print()

print("=== Strategy Distribution ===")
print(f"Total: {stats.get('total', 0)}")
print(f"Paired: {stats.get('paired', 0)}")
print(f"Lost: {stats.get('lost_total', 0)}")
print(f"Rewrite distance exceeded (> {REWRITE_DISTANCE_THRESHOLD}): {stats.get('rewrite_distance_exceeded', 0)}")
print(f"  → Neutral (80%): {stats.get('strategy_neutral', 0)}")
print(f"  → Generated (20%): counted in strategy_generated")
print(f"Strategy generated (total): {stats.get('strategy_generated', 0)}")
print(f"Fallback to neutral: {stats.get('strategy_generated_fallback_to_neutral', 0)}")
print(f"No eligible candidate: {stats.get('no_eligible_candidate', 0)}")
print()
print("Lost sample count:", len(lost_samples))

=== Pair Preview ===
--- Pair 1 ---
Character: 沐雪
Reject source: generated_candidate
Reject strategy: generated
Reject rewrite ratio: 0.7619047619047619
Chosen-neutral structural similarity: 0.8
Chosen output: <think>她依天真语义选择发问，极高碎片率与疑问率使句式松散，最后用“这个嘛”及省略号、括号动作展现呆萌可爱的身份特征。</think>

这个嘛...奇奇怪怪的东西指的是什么呢（天真）
Rejected output: <think>作为天真可爱的少女，她会用好奇和俏皮的语调询问，并可能加入一些语气词或感叹号来体现活泼。</think>

“奇奇怪怪的东西”是啥？

--- Pair 2 ---
Character: 沐雪
Reject source: generated_candidate
Reject strategy: generated
Reject rewrite ratio: 0.9032258064516129
Chosen-neutral structural similarity: 0.33766233766233766
Chosen output: <think>她依活力与可爱语义选用夸张比喻，极高碎片率与祈使感加快节奏，最后用“根本”“啦”配合多重感叹号，展现元气与调皮。</think>

不用担心哦，我的身体可是用钛合金打造的，根本不会受伤哦！与其担心我的身体，不如担心你的又小又弱的身体啦！
Rejected output: <think>作为活泼可爱的性格，她会用亲切的语气和感叹词来表达关心。虽然没有特定签名，但高能量和乐观语义使句子充满活力，短句节奏轻快。</think>

不用担心啦！我的身体很结实，不会受伤的。你更应该多关心自己的身体状况了

--- Pair 3 ---
Character: 沐雪
Reject source: generated_candidate
Reject strategy: generated
Reject rewrite ratio: 0.918918918918919
Chosen-n